# Sauda shopping assistant: test on Groq and Gemini

**Before you start:** click the **key icon** on the left (Secrets) and add your keys:

| Name | Value |
|---|---|
| `GEMINI_API_KEY` | your Gemini key |
| `GROQ_API_KEY` | your Groq key |

Turn on **Notebook access** for each. You can add just one.

Then run the cells from top to bottom: press the ▶ button on each cell and wait for it to finish.

**Got an updated version from Claude?** Use **Runtime → Restart session**, then run the cells again from Step 1.

## Step 1: Download and install (about 30 seconds)

In [ ]:
%cd /content
!rm -rf /content/Usman-hassan
!git clone -q --depth 1 -b claude/ecommerce-ai-assistant-t27kqw https://github.com/Usmanhassan7654/Usman-hassan.git
%cd /content/Usman-hassan/sauda-assistant
!pip install -q -r requirements.txt
print("Installed.")

## Step 2: Load your keys

In [ ]:
import os
from google.colab import userdata

found = []
for name in ("GEMINI_API_KEY", "GROQ_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        found.append(name)
print("Keys loaded:", ", ".join(found) if found else "none. Add them in the key icon on the left, turn on Notebook access, then run this cell again.")

## Step 3: Check the app itself (no keys needed)
Should end with **passed**.

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## Step 4: Chat with the assistant here
Change the message in the last line and run the cell again. The conversation continues between runs.

## Step 3b: Check each provider
Shows **OK** or the exact error for Groq and Gemini. If something fails, send Claude what this prints.

In [ ]:
# Change the model names here if you want to try others.
GROQ_MODEL = "openai/gpt-oss-20b"
GEMINI_MODEL = "gemini-flash-latest"

import sys, json
sys.path.insert(0, ".")
for name in [m for m in sys.modules if m == "sauda" or m.startswith("sauda.")]:
    del sys.modules[name]
from sauda import llm
from sauda.tools import TOOLS

def check(provider, model, key_var):
    if not os.environ.get(key_var):
        print(f"{provider}: skipped, no {key_var}")
        return
    m = llm.OpenAICompatModel(provider, model, os.environ[key_var], max_retries=1)
    convo = [{"role": "user", "content": "I need milk. Record it as a need."}]
    try:
        r1 = m.complete("You are a shopping assistant. Use the update_needs tool.", convo, TOOLS)
        print(f"{provider}:{model}  call 1 OK  (tool calls: {[c['name'] for c in r1.tool_calls] or 'none'})")
        if r1.tool_calls:
            convo += [{"role": "assistant", "content": r1.text, "tool_calls": r1.tool_calls}]
            convo += [{"role": "tool", "id": c["id"], "name": c["name"], "content": json.dumps({"ok": True})} for c in r1.tool_calls]
            r2 = m.complete("You are a shopping assistant. Use the update_needs tool.", convo, TOOLS)
            print(f"{provider}:{model}  call 2 OK  (reply: {r2.text[:80]!r})")
    except Exception as e:
        print(f"{provider}:{model}  FAILED:", e)
        try:
            names = [n for n in m.list_models() if "gemini" in n or provider == "groq"]
            print(f"   Models your {provider} key can use:", ", ".join(names[:40]))
        except Exception as e2:
            print("   (could not list models:", e2, ")")

check("groq", GROQ_MODEL, "GROQ_API_KEY")
check("gemini", GEMINI_MODEL, "GEMINI_API_KEY")

In [ ]:
import sys, uuid
sys.path.insert(0, ".")
for name in [m for m in sys.modules if m == "sauda" or m.startswith("sauda.")]:
    del sys.modules[name]          # always use the freshly downloaded code
from sauda import ingest, llm
from sauda.db import Repo
from sauda.agent import Agent

repo = Repo("colab.db")
scope = repo.add_store("demo-mart", "Demo Mart", "demo-khi", "Karachi", api_key="demo-store-key")
if not repo.catalog_stats(scope)["published_products"]:
    ingest.sync(repo, scope, "data/sample_pk_grocery.csv")

GROQ_MODEL = globals().get("GROQ_MODEL", "openai/gpt-oss-20b")
GEMINI_MODEL = globals().get("GEMINI_MODEL", "gemini-flash-latest")
# Which providers to use, in order. Use PROVIDERS = ["gemini"] or ["groq"] to try just one.
PROVIDERS = [p for p, k in (("groq", "GROQ_API_KEY"), ("gemini", "GEMINI_API_KEY")) if os.environ.get(k)]
spec = ",".join(f"{p}:{GROQ_MODEL if p == 'groq' else GEMINI_MODEL}" for p in PROVIDERS)
agent = Agent(repo, model=llm.build(spec))
session = uuid.uuid4().hex
print("Using:", agent.model.name)

def ask(message):
    r = agent.handle(scope, session, message, emit=lambda e: print("  …", e["text"]))
    print("\nAssistant:", r.reply, "\n")
    for card in r.cards:
        if card["type"] == "plan":
            print("Suggested list:")
            for l in card["lines"]:
                print(f"  {l['packs']} x {l['name']}  Rs {l['line_total']:,.0f}")
            print(f"  Total: Rs {card['total']:,.0f}")
            for b in card["budgets"]:
                print(f"  Budget {b['budget']:,.0f}: {'OVER by ' + format(-b['remaining'], ',.0f') if b['over'] else format(b['remaining'], ',.0f') + ' left'}")
        elif card["type"] == "options":
            for g in card["groups"]:
                print(g["label"] + ":")
                for o in g["options"]:
                    print(f"  {o['ref']}. {o['name']}  Rs {o['price']:,.0f}")
        elif card["type"] == "comparison":
            for i in card["items"]:
                print(f"  {i['ref']}. {i['name']}  Rs {i['price']:,.0f}  {i['unit_price']}")
    if r.basket["items"]:
        print(f"Basket: {r.basket['item_count']} items, Rs {r.basket['total']:,.0f}")
    if r.grounding.get("corrected"):
        print("(The assistant corrected its first reply:", r.grounding["corrected"], ")")
    print(f"[{r.usage['model_calls']} model calls via {', '.join(r.usage['models']) or 'none'}]")
    for err in r.usage.get("errors", []):
        print("PROVIDER ERROR (send this to Claude):", err)

ask("chicken biryani banana hai 6 logon ke liye")

In [ ]:
ask("Olpers nahi, aur oil sasta wala do")

In [ ]:
ask("theek hai, sab basket mein daal do")

## Step 5: Run the automatic customer tests
Progress is printed after every answer. **QUICK = True** runs 4 core tests (about 5–10 minutes on free plans); set it to False for all 10 (15–30 minutes on Groq's free plan). Each run uses some API credit.

At the end you'll see a line like **3/4 journey runs passed**. Copy what this cell prints and send it to Claude.

In [ ]:
QUICK = True          # False = all 10 tests
PROVIDERS = ["groq", "gemini"]   # remove one to test only the other

flags = "--quick" if QUICK else ""
for provider in PROVIDERS:
    key = {"groq": "GROQ_API_KEY", "gemini": "GEMINI_API_KEY"}[provider]
    if not os.environ.get(key):
        print(f"Skipping {provider}: no key.")
        continue
    model = GROQ_MODEL if provider == "groq" else GEMINI_MODEL
    print(f"===================== {provider}:{model} =====================")
    !python -u evals/journeys.py --llm {provider}:{model} {flags}

## Step 6 (optional): Open the full chat website
Run this, then click the link that appears.

In [ ]:
import subprocess, time
from google.colab import output
env = {**os.environ, "SAUDA_DEMO": "1", "SAUDA_DB": "web.db"}
server = subprocess.Popen(["python", "-m", "uvicorn", "sauda.server:app", "--port", "8000"], env=env,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
output.serve_kernel_port_as_window(8000, anchor_text="Open the Sauda chat website")